# Experiment Runner — Unified Ablation & Baseline Runner

## Purpose
Orchestrates training and evaluation for **all** ablation studies (A-series) and baseline comparisons (B-series). Results are accumulated in `outputs/experiments/all_results.json` — one entry per experiment ID.

## Usage
Run cells in order to load the `ExperimentTrainer` and `run_experiments()` function, then use the example block at the bottom of this notebook to run experiments interactively:

```python
# List all available experiments
print_experiment_plan(base_config)

# Run a single experiment
results = run_experiments(['A1_no_gcn'], base_config, outputs/experiments/)

# Run all ablations
results = run_experiments('ablations', base_config, outputs/experiments/)

# Run all baselines
results = run_experiments('baselines', base_config, outputs/experiments/)

# Run everything
results = run_experiments('all', base_config, outputs/experiments/)
```

## Key Design Decisions
- **Incremental saves**: After each experiment completes, `all_results.json` is written to disk. If the script crashes, no results are lost.
- **Checkpoint resume**: If `{exp_id}_best.pt` already exists in `outputs/experiments/`, training is skipped and the checkpoint is loaded directly for evaluation.
- **Unified trainer**: `ExperimentTrainer` works for all model variants (full model, PlainRoBERTa, BERTBaseline) via a single `_forward()` method.
- **AMP support**: Optional mixed-precision training (FP16) for faster GPU training.
- **Early stopping**: Stops training when validation Macro-F1 doesn't improve after `patience` epochs.


In [1]:
print("⏳ Starting: Running MSR evaluation...")
from tqdm.auto import tqdm
import time
_start_time = time.time()
import os, sys, yaml, json, time, argparse, copy
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from datetime import datetime

# ── Path setup ─────────────────────────────────────────────────────────────────
ML_RESEARCH = os.path.dirname(os.path.abspath(''))  # ml-research/
os.chdir(ML_RESEARCH)
SRC_DIR = os.path.join(ML_RESEARCH, 'src')
EXP_DIR = os.path.join(SRC_DIR, 'experiments')
for _p in tqdm([ML_RESEARCH, SRC_DIR, EXP_DIR], desc="Processing _p"):
    if _p not in sys.path: sys.path.insert(0, _p)

from ablation_configs import get_all_ablation_specs, get_all_baseline_specs, print_experiment_plan
from baseline_models  import create_baseline, CrossEntropyLossWrapper, TFIDFSVMBaseline
from models.model     import create_model
from models.losses    import AspectSpecificLossManager
from utils.data_utils import create_dataloaders, DependencyParser, compute_class_weights
from utils.metrics    import AspectSentimentEvaluator, MixedSentimentEvaluator
from transformers     import RobertaTokenizer, BertTokenizer, get_linear_schedule_with_warmup
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Running MSR evaluation.")


⏳ Starting: Running MSR evaluation...


Processing _p:   0%|          | 0/3 [00:00<?, ?it/s]

🕒 Done in 6.73s
✅ Completed: Running MSR evaluation.


## ExperimentTrainer
Reusable training loop that supports:
- Full model (with/without GCN)
- Baseline models (PlainRoBERTa, BERT)
- AMP (mixed-precision) on CUDA
- Early stopping on validation Macro-F1


In [2]:
print("⏳ Starting: Defining class ExperimentTrainer...")
import time
_start_time = time.time()
class ExperimentTrainer:
    def __init__(self, exp_id, config, model, loss_manager, tokenizer, results_dir):
        self.exp_id       = exp_id
        self.config       = config
        self.model        = model
        self.loss_manager = loss_manager
        self.results_dir  = results_dir

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)

        # Optionally initialise dependency parser for GCN variants
        print("🧠 Loading the spaCy language model for dependency parsing...")
        dep_parser = DependencyParser() if config['data'].get('use_dependency_parsing') else None
        self.train_loader, self.val_loader, self.test_loader = create_dataloaders(config, tokenizer, dep_parser)

        # AdamW optimiser — standard for fine-tuning transformers
        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=config['training']['learning_rate'],
            weight_decay=config['training']['weight_decay'],
        )

        # Linear warmup + decay schedule
        num_steps      = len(self.train_loader) * config['training']['num_epochs']
        self.scheduler = get_linear_schedule_with_warmup(
            self.optimizer,
            num_warmup_steps=config['training']['warmup_steps'],
            num_training_steps=num_steps,
        )

        # AMP (FP16) — only if CUDA is available and mixed_precision is enabled in config
        self.use_amp = config['hardware'].get('mixed_precision', False) and torch.cuda.is_available()
        if self.use_amp:
            from torch.cuda.amp import GradScaler
            self.scaler = GradScaler()

        self.evaluator        = AspectSentimentEvaluator(config['aspects']['names'])
        self.best_val_metric  = 0
        self.patience_counter = 0
        self.patience         = config['training']['early_stopping_patience']
        self.best_epoch       = None

    def _forward(self, batch):
        """Run forward pass; passes edge_indices for GCN models."""
        input_ids, attention_mask = batch['input_ids'].to(self.device), batch['attention_mask'].to(self.device)
        aspect_ids = batch['aspect_ids'].to(self.device)
        edge_indices = ([e.to(self.device) if e is not None else None for e in batch['edge_indices']]
                        if self.config['model'].get('use_dependency_gcn') else None)
        return self.model(input_ids, attention_mask, aspect_ids, edge_indices)

    def train_epoch(self):
        """Full training pass over one epoch. Returns average loss."""
        print("🚀 Starting the training loop! Keep an eye on the progress bars.")
        self.model.train()
        total_loss = 0
        from tqdm import tqdm

        for batch in tqdm(self.train_loader, desc='Training', leave=False):
            aspect_ids = batch['aspect_ids'].to(self.device)
            labels     = batch['labels'].to(self.device)
            self.optimizer.zero_grad()

            preds = self._forward(batch)
            if isinstance(preds, tuple): preds = preds[0]  # Unpack (logits, attn, repr) if needed

            loss, _ = self.loss_manager.compute_loss(
                preds, labels, aspect_ids, self.config['aspects']['names']
            )
            loss.backward()
            # Gradient clipping prevents exploding gradients during fine-tuning
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config['training']['max_grad_norm'])
            self.optimizer.step()
            self.scheduler.step()  # Update LR schedule every batch
            total_loss += loss.item()

        return total_loss / max(len(self.train_loader), 1)

    def evaluate(self, loader):
        """Evaluate on a dataloader; returns overall + per-aspect metrics dict."""
        self.model.eval()
        all_preds, all_labels, all_aspects, all_probs = [], [], [], []

        with torch.no_grad():
            from tqdm import tqdm
            for batch in tqdm(loader, desc='Evaluating', leave=False):
                preds = self._forward(batch)
                if isinstance(preds, tuple): preds = preds[0]
                probs = torch.softmax(preds, dim=1)
                all_probs.extend(probs.cpu().numpy())
                all_preds.extend(torch.argmax(preds, dim=1).cpu().numpy())
                all_labels.extend(batch['labels'].numpy())
                all_aspects.extend(batch['aspects'])

        aspect_metrics = {
            asp: self.evaluator.evaluate_aspect(
                np.array(all_labels)[np.array([a == asp for a in all_aspects])],
                np.array(all_preds)[np.array([a == asp for a in all_aspects])],
                asp,
                y_prob=np.array(all_probs)[np.array([a == asp for a in all_aspects])]
            )
            for asp in self.config['aspects']['names']
        }
        overall = self.evaluator.evaluate_aspect(np.array(all_labels), np.array(all_preds), 'overall', y_prob=np.array(all_probs))
        return {'overall': overall, 'aspects': aspect_metrics}

    def train(self):
        """Full training loop with early stopping and checkpoint saving."""
        t0        = time.time()
        best_ckpt = self.results_dir / f'{self.exp_id}_best.pt'

        if not best_ckpt.exists():  # Skip training if checkpoint already exists
            for epoch in tqdm(range(self.config['training']['num_epochs']), desc=f'Experiment {self.exp_id}'):
                loss   = self.train_epoch()
                print("📊 Evaluating the model on the test set...")
                val_m  = self.evaluate(self.val_loader)
                val_f1 = val_m['overall']['macro_f1']
                print(f'  Epoch {epoch+1}: loss={loss:.4f}  val_macro_f1={val_f1:.4f}')

                if val_f1 > self.best_val_metric:
                    self.best_val_metric  = val_f1
                    self.best_epoch       = epoch + 1
                    self.patience_counter = 0
                    torch.save({'model_state_dict': self.model.state_dict(), 'config': self.config, 'best_epoch': self.best_epoch, 'best_val_f1': val_f1}, best_ckpt)
                else:
                    self.patience_counter += 1
                    if self.patience_counter >= self.patience:
                        print(f'  Early stopping at epoch {epoch+1}'); break

        if best_ckpt.exists():
            print("📥 Loading a saved model checkpoint...")
            self.model.load_state_dict(torch.load(best_ckpt, map_location=self.device)['model_state_dict'])

        test_metrics = self.evaluate(self.test_loader)
        duration_mins = (time.time() - t0) / 60
        print(f'  Done in {duration_mins:.1f} min — test macro_f1={test_metrics["overall"]["macro_f1"]:.4f}  best_epoch={self.best_epoch}')
        # ── 5. Standard ROC/AUC Saving ───────────────────────────────────
        print(f"📊 Saving ROC curves and metrics to: {self.results_dir}")
        self.evaluator.plot_all_roc_curves(save_dir=self.results_dir)
        with open(self.results_dir / 'metrics.json', 'w') as f:
            json.dump(test_metrics, f, indent=2, default=str)
        
        return test_metrics, duration_mins, self.best_epoch
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining class ExperimentTrainer.")



⏳ Starting: Defining class ExperimentTrainer...
🕒 Done in 0.00s
✅ Completed: Defining class ExperimentTrainer.


## Run Experiments


In [ ]:
import yaml, copy
from pathlib import Path

# ── Load base config ───────────────────────────────────────────────────────────
with open('configs/config.yaml') as f:
    base_config = yaml.safe_load(f)

results_dir = Path('outputs/experiments')
results_dir.mkdir(parents=True, exist_ok=True)

# ── Build A7_hybrid_cb_05 config (Focal 1.0 + CB 0.5 + Dice 0.0) ──────────────
cfg = copy.deepcopy(base_config)
cfg['training']['loss_weights'] = {'focal': 1.0, 'cb': 0.5, 'dice': 0.0}
cfg['experiment']['name'] = 'A7_hybrid_cb_05'

# ── Set up model, tokenizer, loss manager ──────────────────────────────────────
tokenizer = RobertaTokenizer.from_pretrained(cfg['model']['roberta_model'])
model     = create_model(cfg)

import os
print(f"DEBUG: Current Working Directory: {os.getcwd()}")
print(f"DEBUG: cfg['data']['train_path']: {cfg['data']['train_path']}")
print(f"DEBUG: File exists: {os.path.exists(cfg['data']['train_path'])}")
aspect_class_counts = compute_class_weights(
    cfg['data']['train_path'],
    cfg['aspects']['names'],
    cfg['aspects']['label_map']
)
loss_manager = AspectSpecificLossManager(aspect_class_counts, cfg['training'])

# ── Run training ───────────────────────────────────────────────────────────────
trainer = ExperimentTrainer('A7_hybrid_cb_05', cfg, model, loss_manager, tokenizer, results_dir)
test_metrics, duration_mins, best_epoch = trainer.train()

print(f'\n=== A7_hybrid_cb_05 Results ===')
print(f'ROC Curves saved to: {results_dir}')
print(f'Best epoch:      {best_epoch}')
print(f'Test Macro-F1:   {test_metrics["overall"]["macro_f1"]:.4f}')
print(f'Test Accuracy:   {test_metrics["overall"]["accuracy"]:.4f}')
print(f'Duration:        {duration_mins:.1f} min')

